# qlib baseline vs. LLM agents (ICAIF 2026 setup)

**Question.** The LLM agents of the paper are costly. Can a traditional quantitative pipeline —
[qlib](https://github.com/microsoft/qlib) features + LightGBM — reach a similar **trading return**
on the **same 9 stocks, same 24 monthly decision dates (Jan 2024 – Dec 2025) and same evaluation**?

**Models** (see `qlib_baseline/README.md`):

| Model | Features |
|---|---|
| `price` | Alpha158 price/volume factors (157 — VWAP is not in the B3 bundle) |
| `fundamentals` | Alpha158 + 19 bundle fields equivalent to the paper's fundamental indicators (P/L, P/VP, EV/EBITDA, ROE, ROIC, margins, …) |

Each model is retrained every month with data available on **D-1** (labels purged so no future
return leaks) and trained on two universes: `broad` (point-in-time B3 liquidity universe + the 9
stocks) and `paper9` (only the 9 stocks). The predicted 1-month return is turned into
Comprar / Vender / Manter by two rules: `threshold` (sign of the prediction, τ = 0) and `rank`
(top-3 buy / bottom-3 sell among the 9).

**Evaluation** is the paper's: trades at the COTAHIST close of the decision day; the main metric is
the paper's *trading return* **Norm. Return** = Σ(sell − avg buy) / Σ avg buy, plus Sharpe
(SELIC 15%). The functions in `qlib_baseline/evaluation.py` are copied verbatim from
`manager_evaluation.ipynb`, and the LLM rows below must match Table 1 of the paper.

Run with the qlib environment (Python 3.12):
```bash
cd qlib_baseline && uv sync && cd ..
qlib_baseline/.venv/bin/jupyter lab qlib_evaluation.ipynb
```

In [1]:
import os
import sys
import warnings

warnings.filterwarnings("ignore")
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"
sys.path.insert(0, os.path.abspath("qlib_baseline"))

# ruff: noqa: E402

import pandas as pd
from tabulate import tabulate

import config
import decisions
import evaluation as ev
import walk_forward as wf

# False: reuse the committed predictions in qlib_baseline/output (seconds).
# True: retrain all 4 models x 24 months from data/qlib_data (a few minutes on a laptop).
RETRAIN = False

DATES = config.decision_dates()
print(f"{len(DATES)} decision dates: {DATES[0]} … {DATES[-1]}")
print("Stocks:", config.STOCKS)

24 decision dates: 2024-01-02 … 2025-12-01
Stocks: ['ALUP11', 'AURE3', 'CPLE3', 'EALT4', 'ENEV3', 'EQTL3', 'PETR4', 'RECV3', 'VALE3']


## 1. Walk-forward training (monthly, D-1)

In [2]:
if RETRAIN:
    import shutil

    shutil.rmtree(wf.OUTPUT_DIR, ignore_errors=True)
wf.init_qlib()
preds = wf.run_all(config.FEATURE_SETS, config.UNIVERSES, DATES)
example = preds[("fundamentals", "broad")]
example.head(9)

Cache hit: /Users/thiagocastroferreira/Documents/workspace/workflow-vs-agent-fundamentals-br/qlib_baseline/output/predictions_price_broad.csv


Cache hit: /Users/thiagocastroferreira/Documents/workspace/workflow-vs-agent-fundamentals-br/qlib_baseline/output/predictions_price_paper9.csv


Cache hit: /Users/thiagocastroferreira/Documents/workspace/workflow-vs-agent-fundamentals-br/qlib_baseline/output/predictions_fundamentals_broad.csv


Cache hit: /Users/thiagocastroferreira/Documents/workspace/workflow-vs-agent-fundamentals-br/qlib_baseline/output/predictions_fundamentals_paper9.csv


,analysis_date,feature_date,stock_id,score,label,train_end,n_train,best_iteration
0,2024-01-02,2023-12-28,ALUP11,0.009531,-0.044335,2023-11-27,138297,75
1,2024-01-02,2023-12-28,AURE3,0.008123,-0.009022,2023-11-27,138297,75
2,2024-01-02,2023-12-28,CPLE3,0.008069,-0.031318,2023-11-27,138297,75
3,2024-01-02,2023-12-28,EALT4,0.010889,0.069547,2023-11-27,138297,75
4,2024-01-02,2023-12-28,ENEV3,-0.003058,-0.039880,2023-11-27,138297,75
5,2024-01-02,2023-12-28,EQTL3,0.005715,0.012286,2023-11-27,138297,75
6,2024-01-02,2023-12-28,PETR4,0.010132,0.078878,2023-11-27,138297,75
7,2024-01-02,2023-12-28,RECV3,0.007919,0.109774,2023-11-27,138297,75
8,2024-01-02,2023-12-28,VALE3,0.015082,-0.120571,2023-11-27,138297,75


## 2. Signal quality

Per decision date, correlation across the 9 stocks between the predicted and the realised return
from D to D+21 trading days (IC = Pearson, Rank IC = Spearman). *Hit rate* = share of predictions
with the right sign.

In [3]:
def signal_metrics(p: pd.DataFrame) -> dict:
    by_date = p.groupby("analysis_date")
    ic = by_date.apply(lambda g: g.score.corr(g.label))
    ric = by_date.apply(lambda g: g.score.corr(g.label, method="spearman"))
    return {
        "IC": ic.mean(),
        "ICIR": ic.mean() / ic.std(),
        "Rank IC": ric.mean(),
        "Rank ICIR": ric.mean() / ric.std(),
        "Hit rate": ((p.score > 0) == (p.label > 0)).mean(),
        "% positive predictions": (p.score > 0).mean(),
    }


sig = pd.DataFrame({f"{f}/{u}": signal_metrics(p) for (f, u), p in preds.items()}).T
print(tabulate(sig, headers="keys", tablefmt="github", floatfmt=".4f"))

|                     |      IC |    ICIR |   Rank IC |   Rank ICIR |   Hit rate |   % positive predictions |
|---------------------|---------|---------|-----------|-------------|------------|--------------------------|
| price/broad         |  0.1623 |  0.3697 |    0.0792 |      0.1962 |     0.4907 |                   0.9954 |
| price/paper9        | -0.0126 | -0.0324 |    0.0047 |      0.0125 |     0.4722 |                   0.9583 |
| fundamentals/broad  |  0.0694 |  0.1546 |    0.0536 |      0.1450 |     0.4861 |                   0.9815 |
| fundamentals/paper9 |  0.0396 |  0.0978 |   -0.0035 |     -0.0083 |     0.5278 |                   0.9028 |


## 3. Monthly decisions → `results/qlib_*`

Each (features × universe × rule) is written to `results/qlib_<features>_<universe>_<rule>/` with the
same files as the LLM experiments, so it can also be loaded by `manager_evaluation.ipynb`.

In [4]:
QLIB_FOLDERS = {}
for (features, universe), p in preds.items():
    for rule in config.RULES:
        name = f"qlib_{features}_{universe}_{rule}"
        QLIB_FOLDERS[name] = decisions.write_strategy(p, rule, name)
print("\n".join(str(f.relative_to(config.REPO_ROOT)) for f in QLIB_FOLDERS.values()))

results/qlib_price_broad_threshold
results/qlib_price_broad_rank
results/qlib_price_paper9_threshold
results/qlib_price_paper9_rank
results/qlib_fundamentals_broad_threshold
results/qlib_fundamentals_broad_rank
results/qlib_fundamentals_paper9_threshold
results/qlib_fundamentals_paper9_rank


## 4. Trading return vs. the paper (main result)

Paper rows are recomputed from `results/` and must equal Table 1
(B&H −1.35% / 0.2124, +FactsGPT 14.44% / 1.0265, …).

In [5]:
R = config.RESULTS_DIR
PAPER = {
    "Price (LLM)": R / "baseline",
    "Fund.LLM": R / "manager",
    "Fund.Rule": R / "sem_fato_relevante_gpt5mini",
    "+FactsGPT": R / "completo_gpt5mini",
    "+FactsSabiá": R / "completo_sabiazinho4",
    "Hybrid": R / "hibrido_fatos_sabiazinho4_gestor_gpt5mini",
}
STOCKS = set(config.STOCKS)

ref_decisions, ref_results = ev.load(PAPER["Fund.Rule"])
N_DAYS = ev.n_days([d for d in ref_decisions if d["stock_id"] in STOCKS])

returns = {"B&H": ev.compute_buy_and_hold(ref_results, STOCKS)}
# Reference with no model at all: "Comprar" every month for every stock (monthly DCA).
always_buy = [dict(d, recommendation="Comprar") for d in ref_decisions]
returns["Always Comprar (DCA)"] = ev.compute_portfolio_returns(always_buy, ref_results, STOCKS)
for name, folder in {**PAPER, **QLIB_FOLDERS}.items():
    returns[name] = ev.compute_portfolio_returns(*ev.load(folder), STOCKS)

summary = pd.DataFrame([ev.summarize(n, d, N_DAYS) for n, d in returns.items()]).set_index(
    "Strategy"
)
summary["Source"] = [
    "qlib" if n.startswith("qlib_") else "reference" if n.startswith("Always") else "paper"
    for n in summary.index
]
show = summary.copy()
for c in ["Norm. Return", "Trading Return", "Avg/Stock", "Win rate"]:
    show[c] = (show[c] * 100).map("{:.2f}%".format)
show["Sharpe"] = show["Sharpe"].map("{:.4f}".format)
print(f"{N_DAYS} calendar days between first and last decision")
print(tabulate(show, headers="keys", tablefmt="github"))

699 calendar days between first and last decision
| Strategy                           | Norm. Return   |   Sharpe | Trading Return   | Avg/Stock   |   # Trades | Win rate   | Source    |
|------------------------------------|----------------|----------|------------------|-------------|------------|------------|-----------|
| B&H                                | -1.35%         |   0.2124 | 77.27%           | 8.59%       |          9 | 55.56%     | paper     |
| Always Comprar (DCA)               | 8.89%          |   0.4638 | 102.32%          | 11.37%      |          9 | 77.78%     | reference |
| Price (LLM)                        | 18.04%         |   1.7188 | 89.81%           | 9.98%       |          4 | 100.00%    | paper     |
| Fund.LLM                           | 9.50%          |   0.5215 | 150.55%          | 16.73%      |          9 | 77.78%     | paper     |
| Fund.Rule                          | 6.74%          |   0.1875 | 63.33%           | 7.04%       |          8 | 75.00%   

In [6]:
# Sanity check: the evaluation is identical to the paper's notebook
assert f"{summary.loc['B&H', 'Norm. Return'] * 100:.2f}" == "-1.35"
assert f"{summary.loc['+FactsGPT', 'Norm. Return'] * 100:.2f}" == "14.44"
assert f"{summary.loc['+FactsGPT', 'Sharpe']:.4f}" == "1.0265"
print("✓ paper rows match Table 1")

✓ paper rows match Table 1


## 5. Sensitivity of the threshold rule

With τ = 0 a model that predicts positive returns for every stock degenerates into
"buy every month". Raising τ asks for a minimum expected 1-month return before buying
(≈ 1% is the monthly SELIC in 2024–2025) and sells only on clearly negative predictions.

In [7]:
rows = []
for (features, universe), p in preds.items():
    for tau in [0.0, 0.005, 0.01, 0.02]:
        dec = decisions.make_decisions(p, "threshold", tau=tau)
        data = ev.compute_portfolio_returns(dec, ref_results, STOCKS)
        s = ev.summarize(f"{features}/{universe}", data, N_DAYS)
        rows.append(
            [
                s["Strategy"],
                tau,
                f"{s['Norm. Return'] * 100:.2f}%",
                f"{s['Sharpe']:.4f}",
                s["# Trades"],
            ]
        )
print(
    tabulate(rows, headers=["Model", "τ", "Norm. Return", "Sharpe", "# Trades"], tablefmt="github")
)

| Model               |     τ | Norm. Return   |   Sharpe |   # Trades |
|---------------------|-------|----------------|----------|------------|
| price/broad         | 0     | 8.82%          |   0.5309 |         10 |
| price/broad         | 0.005 | 8.89%          |   0.4691 |          9 |
| price/broad         | 0.01  | 7.82%          |   0.5305 |          9 |
| price/broad         | 0.02  | 0.00%          | nan      |          0 |
| price/paper9        | 0     | 3.79%          |   0.2686 |         17 |
| price/paper9        | 0.005 | 6.01%          |   0.4546 |         13 |
| price/paper9        | 0.01  | 5.06%          |   0.2884 |         13 |
| price/paper9        | 0.02  | 8.60%          |   0.5307 |          9 |
| fundamentals/broad  | 0     | 7.31%          |   0.3248 |         10 |
| fundamentals/broad  | 0.005 | 8.77%          |   0.4417 |          9 |
| fundamentals/broad  | 0.01  | 0.95%          |   0.081  |          6 |
| fundamentals/broad  | 0.02  | 55.27%         | na

## 6. Per-stock Norm. Return

In [8]:
best_qlib = summary[summary.Source == "qlib"]["Norm. Return"].idxmax()
cols = ["B&H", "Price (LLM)", "Fund.Rule", "+FactsGPT", "Hybrid", best_qlib] + [
    n for n in QLIB_FOLDERS if n != best_qlib
]
per_stock = pd.DataFrame(
    {c: {s: ev.stock_return_norm(returns[c], s) for s in config.STOCKS} for c in cols}
)
per_stock.loc["TOTAL"] = [ev.total_return_norm(returns[c]) for c in cols]
print(f"Best qlib strategy: {best_qlib}")
print(tabulate((per_stock * 100).round(2), headers="keys", tablefmt="github"))

Best qlib strategy: qlib_price_broad_threshold
|        |    B&H |   Price (LLM) |   Fund.Rule |   +FactsGPT |   Hybrid |   qlib_price_broad_threshold |   qlib_price_broad_rank |   qlib_price_paper9_threshold |   qlib_price_paper9_rank |   qlib_fundamentals_broad_threshold |   qlib_fundamentals_broad_rank |   qlib_fundamentals_paper9_threshold |   qlib_fundamentals_paper9_rank |
|--------|--------|---------------|-------------|-------------|----------|------------------------------|-------------------------|-------------------------------|--------------------------|-------------------------------------|--------------------------------|--------------------------------------|---------------------------------|
| ALUP11 |  12.25 |         17.99 |       13.82 |       14.09 |    13.82 |                        13.82 |                    1.09 |                          4.43 |                    -1.85 |                               13.82 |                          -2.51 |                      

## 7. Decision timeline — best qlib strategy vs. +FactsGPT (C = Comprar, V = Vender, · = Manter, ? = no decision)

In [9]:
def timeline(folder) -> pd.DataFrame:
    dec = pd.DataFrame(ev.load(folder)[0])
    dec["r"] = dec.recommendation.map({"Comprar": "C", "Vender": "V", "Manter": "·"}).fillna("?")
    t = dec.pivot(index="stock_id", columns="analysis_date", values="r")
    t.columns = [c[2:7] for c in t.columns]
    return t.loc[config.STOCKS]


for name, folder in [(best_qlib, QLIB_FOLDERS[best_qlib]), ("+FactsGPT", PAPER["+FactsGPT"])]:
    print(name)
    print(tabulate(timeline(folder), headers="keys", tablefmt="simple"))
    print()

qlib_price_broad_threshold
stock_id    24-01    24-02    24-03    24-04    24-05    24-06    24-07    24-08    24-09    24-10    24-11    24-12    25-01    25-02    25-03    25-04    25-05    25-06    25-07    25-08    25-09    25-10    25-11    25-12
----------  -------  -------  -------  -------  -------  -------  -------  -------  -------  -------  -------  -------  -------  -------  -------  -------  -------  -------  -------  -------  -------  -------  -------  -------
ALUP11      C        C        C        C        C        C        C        C        C        C        C        C        C        C        C        C        C        C        C        C        C        C        C        C
AURE3       C        C        C        C        C        C        C        V        C        C        C        C        C        C        C        C        C        C        C        C        C        C        C        C
CPLE3       C        C        C        C        C        C        C        C   

## 8. Findings (with the committed predictions)

* **No qlib configuration reaches the best LLM agents.** The best one, `price/broad/threshold`, gets
  8.82% Norm. Return / Sharpe 0.53. That is below +FactsGPT (14.44% / 1.03) and Price-LLM
  (18.04% / 1.72), roughly at the level of Fund.LLM (9.50%) and Hybrid (9.78%), and above Fund.Rule
  (6.74%) and B&H (−1.35%).
* **Most of that return is "always buy", not stock selection.** The models predict a positive
  1-month return for 90–99% of stock-months, so with τ = 0 the threshold rule is almost the
  *Always Comprar (DCA)* reference, which needs no model at all and gets 8.89% / 0.46, the same as
  the best qlib strategy. See also the timeline: the best qlib
  strategy sells only once in 216 decisions. The LLM agents are much more selective: they stay out of
  AURE3/ENEV3 in 2024, sell EQTL3 early, and sell CPLE3 at the end.
* **The cross-sectional signal is weak.** The best Rank IC (price/broad) is 0.08 with a 49% hit
  rate. The rank rule (forced buy/sell of 3 stocks every month) is the purest test of stock
  selection, and it stays around 0% Norm. Return for every model.
* **Universe matters more than fundamentals.** Training on the broad universe (~100 liquid stocks
  per date) gives a better signal (IC 0.16) than training on the 9 stocks alone (IC ≈ 0). Adding the
  19 fundamentals does not help on the broad universe. It helps on `paper9` with τ ≥ 1%
  (fundamentals/paper9 reaches 12.59% / 0.64 with τ = 2%), but τ is chosen here in hindsight, so
  that number is optimistic.

The LLM agents' edge therefore looks like it comes from deciding **when not to hold** a stock, which
monthly LightGBM return forecasts on price/fundamental factors do not capture. Natural next steps:
extra signals (e.g. environmental factors, see `qlib_baseline/README.md`), a classification label
(beat SELIC or not), and choosing τ on a validation period.